In [0]:
# Ejecutar solo si es necesario
spark.sql("DROP TABLE IF EXISTS catalog_cemm_expl_bcp_prod.bcp_expl_mmgr_mlde.bcp_scoring_app_troncal_bhv_pyme_dev")

In [0]:
%run ../config/config

In [0]:
# =============================================================================
# 2. IMPORTACIÓN DE LIBRERÍAS
# =============================================================================

import logging
import os
import sys
import time

from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, StringType

sys.path.insert(0, str(PROJECT_PATH))
from utils.logger import MLOpsLogger, log_dataframe_info

In [0]:
# =============================================================================
# 3. CONFIGURACIÓN DE LOGGING
# =============================================================================

logger = MLOpsLogger(
    name='08_write_lhcl',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO',
    log_dir=LOGS_PATH,
    is_job_run=True,
    job_context={
        'mes_vta': MES_VTA,
        'environment': ENV,
        'notebook': '08_write_lhcl'
    }
)

In [0]:
# =============================================================================
# 4. LÓGICA PRINCIPAL: ESCRITURA DE DATOS EN LHCL (TODOS LOS MESES)
# =============================================================================
try:
    logger.info("=" * 60)
    logger.info(f"ESCRITURA LHCL - {ENV.upper()} - TODOS LOS MESES")
    logger.info("=" * 60)
    logger.info(f"Tabla origen: {OUTPUT_TABLE}")
    logger.info(f"Tabla destino: {LHCL_OUTPUT_TABLE}")

    start_time = time.time()

    # 1. DESCUBRIR TODOS LOS MESES DISPONIBLES EN OUTPUT_TABLE
    logger.info("Consultando meses disponibles en tabla de scoring...")
    meses_rows = (
        spark.sql(f"SELECT DISTINCT codmes FROM {OUTPUT_TABLE} ORDER BY codmes")
        .collect()
    )

    meses_disponibles = sorted([int(row.codmes) for row in meses_rows])
    if not meses_disponibles:
        raise ValueError(f"No hay datos en {OUTPUT_TABLE}")

    logger.info(f"Meses encontrados ({len(meses_disponibles)}): {meses_disponibles}")

    # 2. VERIFICAR SI LA TABLA LHCL DESTINO YA EXISTE
    try:
        spark.read.table(LHCL_OUTPUT_TABLE).limit(0)
        table_exists = True
    except:
        table_exists = False
    logger.info(f"Tabla LHCL destino existe: {table_exists}")

    # 3. ITERAR SOBRE CADA MES
    resultados = []

    for idx_mes, mes_actual in enumerate(meses_disponibles, 1):
        logger.info("")
        logger.info("-" * 60)
        logger.info(f"MES {idx_mes}/{len(meses_disponibles)}: codmes={mes_actual}")
        logger.info("-" * 60)

        try:
            mes_start = time.time()

            # 3.1 LEER DATOS DEL MES
            df_scoring = spark.sql(
                f"SELECT * FROM {OUTPUT_TABLE} WHERE codmes = {mes_actual}"
            )
            record_count = df_scoring.count()
            logger.info(f" Registros encontrados: {record_count:,}")

            if record_count == 0:
                logger.warning(f" Sin datos para codmes={mes_actual}, saltando...")
                resultados.append({'mes': mes_actual, 'registros': 0, 'exitoso': False, 'error': 'Sin datos'})
                continue

            # 3.2 NORMALIZAR NOMBRES DE COLUMNA A LOWERCASE
            df_scoring = df_scoring.toDF(*[c.lower() for c in df_scoring.columns])

            # 3.3 CONSTRUIR COLUMNAS FIJAS (nivel raíz)
            select_exprs = []
            source_cols_used = set()

            for dest_col, src_col in LHCL_FIXED_COLUMNS.items():
                source_cols_used.add(src_col)
                if src_col in df_scoring.columns:
                    if dest_col == "numpd":
                        select_exprs.append(F.col(src_col).cast(DecimalType(19, 8)).alias(dest_col))
                    elif dest_col in ("codclaveunicocli", "codclavepartycli"):
                        select_exprs.append(F.col(src_col).cast(StringType()).alias(dest_col))
                    elif dest_col == "codmes":
                        select_exprs.append(F.col(src_col).cast("int").alias(dest_col))
                    else:
                        select_exprs.append(F.col(src_col).alias(dest_col))
                else:
                    select_exprs.append(F.lit(None).cast(StringType()).alias(dest_col))
                    logger.warning(f"   {dest_col} <- NULL (columna '{src_col}' no encontrada)")

            # 3.4 GENERAR COLUMNAS DE METADATA
            select_exprs.append(F.current_date().alias("fecrutina"))
            select_exprs.append(F.current_timestamp().alias("fecactualizacionregistro"))

            # 3.5 CONSTRUIR STRUCT colvariablesmodeloanalitico CON CASTEO
            struct_cols = [c for c in df_scoring.columns if c not in source_cols_used]

            if struct_cols:
                struct_fields = []
                for c in struct_cols:
                    target_type = LHCL_STRUCT_TYPES.get(c)
                    if target_type:
                        if target_type == "string":
                            src_type = df_scoring.schema[c].dataType.simpleString()
                            if src_type in ("double", "float"):
                                struct_fields.append(F.col(c).cast("int").cast("string").alias(c))
                            else:
                                struct_fields.append(F.col(c).cast("string").alias(c))
                        else:
                            struct_fields.append(F.col(c).cast(target_type).alias(c))
                    else:
                        struct_fields.append(F.col(c).alias(c))
                select_exprs.append(
                    F.struct(struct_fields).alias("colvariablesmodeloanalitico")
                )
            else:
                select_exprs.append(
                    F.lit(None).alias("colvariablesmodeloanalitico")
                )

            df_lhcl = df_scoring.select(*select_exprs)

            # 3.6 LOG SCHEMA (solo para el primer mes)
            if idx_mes == 1:
                logger.info("  Schema de salida n985:")
                for field in df_lhcl.schema.fields:
                    logger.info(f"    {field.name}: {field.dataType.simpleString()}")

            # 3.7 ESCRIBIR EN LHCL
            if table_exists:
                logger.info(f"  Escribiendo con replaceWhere codmes={mes_actual}...")
                df_lhcl.write \
                    .format("delta") \
                    .mode("overwrite") \
                    .option("replaceWhere", f"codmes = {mes_actual}") \
                    .option("mergeSchema", "true") \
                    .saveAsTable(LHCL_OUTPUT_TABLE)
            else:
                logger.info(f"  Creando tabla {LHCL_OUTPUT_TABLE} por primera vez...")
                df_lhcl.write \
                    .mode("overwrite") \
                    .format("delta") \
                    .partitionBy("codmes") \
                    .saveAsTable(LHCL_OUTPUT_TABLE)
                table_exists = True  # Para los siguientes meses usar replaceWhere

            # 3.8 VERIFICACIÓN DEL MES
            verify_count = spark.sql(
                f"SELECT COUNT(*) as cnt FROM {LHCL_OUTPUT_TABLE} WHERE codmes = {mes_actual}"
            ).collect()[0]["cnt"]

            mes_duration = time.time() - mes_start

            if verify_count != record_count:
                logger.warning(f" ⚠️ Diferencia en conteo: origen={record_count:,}, destino={verify_count:,}")

            logger.info(f" ✓ codmes={mes_actual}: {verify_count:,} registros escritos ({mes_duration:.1f}s)")
            resultados.append({'mes': mes_actual, 'registros': verify_count, 'exitoso': True})

        except Exception as e_mes:
            logger.error(f" X Error en codmes={mes_actual}: {str(e_mes)}")
            resultados.append({'mes': mes_actual, 'registros': 0, 'exitoso': False, 'error': str(e_mes)})
            continue

    # 4. RESUMEN FINAL
    duration = time.time() - start_time
    exitosos = [r for r in resultados if r['exitoso']]
    fallidos = [r for r in resultados if not r['exitoso']]
    total_registros = sum(r['registros'] for r in exitosos)

    logger.info("")
    logger.info("=" * 60)
    logger.info("RESUMEN ESCRITURA LHCL")
    logger.info("=" * 60)
    logger.info(f"  Tabla destino:   {LHCL_OUTPUT_TABLE}")
    logger.info(f"  Meses procesados: {len(resultados)}")
    logger.info(f"  Meses exitosos:   {len(exitosos)}")
    logger.info(f"  Meses fallidos:   {len(fallidos)}")
    logger.info(f"  Total registros:  {total_registros:,}")
    logger.info(f"  Duración total:   {duration:.1f}s")

    for r in exitosos:
        logger.info(f"  ✓ codmes={r['mes']}: {r['registros']:,} registros")
    for r in fallidos:
        logger.warning(f"  X codmes={r['mes']}: {r.get('error', 'desconocido')}")

    logger.info("=" * 60)

    if fallidos:
        logger.warning(f"Hubo {len(fallidos)} mes(es) con error. Revisar logs.")

except Exception as e:
    logger.error(f"Error en escritura LHCL: {str(e)}")
    raise

finally:
    if 'logger' in locals():
        logger.close()
    MLOpsLogger.cleanup_job_timestamp()

In [0]:
print(f"LHCL_OUTPUT_TABLE: '{LHCL_OUTPUT_TABLE}'")

# Obtener los valores únicos de codmes de la tabla especificada
df_codmes = spark.sql(
    """
    SELECT codmes, COUNT(codclaveunicocli) as registros
    FROM catalog_cemm_expl_bcp_prod.bcp_expl_mmgr_mlde.bcp_scoring_app_troncal_bhv_pyme_dev GROUP BY codmes
    ORDER BY codmes
    """
)
display(df_codmes)